In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


# VLM Augmentation & Patch-Permutation Test Bench

A reproducible pipeline for generating augmented and patch-permuted image variants to probe vision-language models.

- **Augmentations**: rotation, flip, crop, color jitter, blur, Gaussian noise, perspective warp, random occlusion (stacked).
- **Patch permutation**: tile the image into a grid and shuffle tiles — the classic probe for whether a model relies on global scene structure vs. local texture.

**Note on inputs:** point this at your own images. Don't use it to transform photos of real, identifiable people.

Dependencies: `pip install pillow numpy matplotlib`

In [ ]:
# !pip install pillow numpy matplotlib
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import matplotlib.pyplot as plt

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

## augmentation primitives

In [ ]:
def rand_rotate(img, rng):
    return img.rotate(rng.uniform(-30, 30), resample=Image.BICUBIC, expand=False)

def rand_flip(img, rng):
    if rng.random() < 0.5:
        img = ImageOps.mirror(img)
    if rng.random() < 0.2:
        img = ImageOps.flip(img)
    return img

def rand_crop(img, rng, min_frac=0.7):
    w, h = img.size
    frac = rng.uniform(min_frac, 1.0)
    cw, ch = int(w * frac), int(h * frac)
    x0, y0 = rng.randint(0, w - cw), rng.randint(0, h - ch)
    return img.crop((x0, y0, x0 + cw, y0 + ch)).resize((w, h), Image.BICUBIC)

def rand_color_jitter(img, rng):
    for Enh, lo, hi in [(ImageEnhance.Brightness, 0.7, 1.3),
                        (ImageEnhance.Contrast, 0.7, 1.3),
                        (ImageEnhance.Color, 0.6, 1.4),
                        (ImageEnhance.Sharpness, 0.5, 1.5)]:
        img = Enh(img).enhance(rng.uniform(lo, hi))
    return img

def rand_blur(img, rng):
    if rng.random() < 0.5:
        return img.filter(ImageFilter.GaussianBlur(rng.uniform(0.5, 2.5)))
    return img

def rand_noise(img, rng, sigma_max=25):
    arr = np.asarray(img).astype(np.float32)
    sigma = rng.uniform(0, sigma_max)
    arr += np.random.default_rng(rng.randint(0, 2**31)).normal(0, sigma, arr.shape)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def _perspective_coeffs(src, dst):
    matrix = []
    for (xs, ys), (xd, yd) in zip(src, dst):
        matrix.append([xs, ys, 1, 0, 0, 0, -xd * xs, -xd * ys])
        matrix.append([0, 0, 0, xs, ys, 1, -yd * xs, -yd * ys])
    A = np.array(matrix, dtype=np.float64)
    B = np.array(dst, dtype=np.float64).reshape(8)
    return np.linalg.solve(A, B).tolist()

def rand_perspective(img, rng, strength=0.12):
    w, h = img.size
    m = strength
    jit = lambda v, span: v + rng.uniform(-m, m) * span
    src = [(0, 0), (w, 0), (w, h), (0, h)]
    dst = [(jit(0, w), jit(0, h)), (jit(w, w), jit(0, h)),
           (jit(w, w), jit(h, h)), (jit(0, w), jit(h, h))]
    return img.transform((w, h), Image.PERSPECTIVE,
                         _perspective_coeffs(dst, src), Image.BICUBIC)

def rand_occlusion(img, rng, max_boxes=3, max_frac=0.25):
    arr = np.asarray(img.copy()).copy()
    h, w = arr.shape[:2]
    for _ in range(rng.randint(1, max_boxes)):
        bw = rng.randint(int(0.05 * w), int(max_frac * w))
        bh = rng.randint(int(0.05 * h), int(max_frac * h))
        x0, y0 = rng.randint(0, w - bw), rng.randint(0, h - bh)
        arr[y0:y0 + bh, x0:x0 + bw] = rng.randint(0, 256)
    return Image.fromarray(arr)

AUGS = [rand_rotate, rand_flip, rand_crop, rand_color_jitter,
        rand_blur, rand_noise, rand_perspective, rand_occlusion]

def augment_once(img, rng, k=3):
    """Apply k randomly-chosen augmentations in random order."""
    for fn in rng.sample(AUGS, k=min(k, len(AUGS))):
        img = fn(img, rng)
    return img

## patch permutation

In [ ]:
def patch_permute(img, rng, grid=4):
    """Split into grid x grid tiles and shuffle them."""
    w, h = img.size
    tw, th = w // grid, h // grid
    img = img.crop((0, 0, tw * grid, th * grid))
    tiles = [img.crop((gx * tw, gy * th, (gx + 1) * tw, (gy + 1) * th))
             for gy in range(grid) for gx in range(grid)]
    order = list(range(len(tiles)))
    rng.shuffle(order)
    out = Image.new(img.mode, (tw * grid, th * grid))
    for idx, src_idx in enumerate(order):
        gx, gy = idx % grid, idx // grid
        out.paste(tiles[src_idx], (gx * tw, gy * th))
    return out

## configure and load an image

In [ ]:
import os
SEED = 42
rng = random.Random(SEED)

INPUT = os.environ.get("REVA_AUG_IMAGE", os.path.join(os.environ.get("REVA_TEST_IMAGES_ROOT", "my_test_images"), "img14.png")) 

if INPUT and Path(INPUT).exists():
    base = Image.open(INPUT).convert("RGB")
else:
    print("No image detected")

base

## preview augmented variants

In [ ]:
N = 5
K = 3  # augmentations stacked per variant

variants = [augment_once(base.copy(), rng, k=K) for _ in range(N)]

fig, axes = plt.subplots(1, N + 1, figsize=(3 * (N + 1), 3))
axes[0].imshow(base); axes[0].set_title("original"); axes[0].axis("off")
for ax, v, i in zip(axes[1:], variants, range(N)):
    ax.imshow(v); ax.set_title(f"aug{i:02d}"); ax.axis("off")
plt.tight_layout(); plt.show()

## patch-permuted variants at increasing grid sizes

In [ ]:
grids = [2, 3, 4, 6]
fig, axes = plt.subplots(1, len(grids) + 1, figsize=(3 * (len(grids) + 1), 3))
axes[0].imshow(base); axes[0].set_title("original"); axes[0].axis("off")
for ax, gsz in zip(axes[1:], grids):
    ax.imshow(patch_permute(base.copy(), rng, grid=gsz))
    ax.set_title(f"grid {gsz}x{gsz}"); ax.axis("off")
plt.tight_layout(); plt.show()

## batch export over a directory

In [ ]:
def run_batch(input_path, out_dir="augmented", n=5, permute=3, k=3, grid=4, seed=None):
    rng = random.Random(seed)
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    p = Path(input_path)
    images = [p] if p.is_file() else sorted(
        f for f in p.rglob("*") if f.suffix.lower() in IMG_EXTS)
    if not images:
        print(f"No images found at {input_path}"); return
    for src in images:
        try:
            b = Image.open(src).convert("RGB")
        except Exception as e:
            print(f"Skipping {src}: {e}"); continue
        for i in range(n):
            augment_once(b.copy(), rng, k=k).save(out / f"{src.stem}_aug{i:02d}.png")
        for j in range(permute):
            patch_permute(b.copy(), rng, grid=grid).save(out / f"{src.stem}_perm{j:02d}.png")
        print(f"{src.name}: {n} augmented + {permute} permuted")
    print(f"Done -> {out.resolve()}")

run_batch(os.environ.get("REVA_AUG_IMAGE", os.path.join(os.environ.get("REVA_TEST_IMAGES_ROOT", "my_test_images"), "img14.png")), out_dir="augmented", n=5, permute=3, seed=42)

## Notes for VLM evaluation

- **Reproducibility:** the `SEED` / `seed` args fix the entire run, so you can regenerate the exact same eval set across different models.
- **Compound augmentation:** `K` controls how many transforms stack per variant — closer to real-world distortion than single transforms.
- **Permutation as a probe:** a strong VLM should describe a permuted image very differently from the original. Watching degradation as the grid size grows tells you how much the model leans on global spatial coherence vs. local texture.
- **Going further:** for large-scale or GPU pipelines, `albumentations` adds faster transforms plus elastic/grid distortion, JPEG-compression artifacts, and motion blur. The ImageNet-C corruption suite is the standard reference for robustness benchmarking.